# 🛒 Retail E-Commerce Chatbot Dataset — Full Analysis & Model Notebook

**Dataset:** `bitext-retail-ecommerce-llm-chatbot-training-dataset.csv`  
**Rows:** 44,884 | **Columns:** 5 (`instruction`, `intent`, `category`, `tags`, `response`)  

### Sections
1. Setup & Imports  
2. Data Loading & Cleaning  
3. Exploratory Data Analysis (EDA)  
4. Matplotlib Static Visualisations  
5. Plotly Interactive Dashboard  
6. Feature Engineering & Preprocessing  
7. Intent Classification Model (TF-IDF + Logistic Regression)  
8. Deep Learning Intent Classifier (optional)  
9. Chatbot (TF-IDF Retrieval + Claude-backed fallback)  
10. Model Evaluation  
11. Save Artefacts


## 1. Setup & Imports

In [ ]:

import warnings, re, os, json, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score, ConfusionMatrixDisplay)
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE

import joblib
import pickle

# Plotly theme
pio.templates.default = "plotly_dark"
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "figure.facecolor": "#1a1a2e",
                      "axes.facecolor": "#16213e", "axes.labelcolor": "white",
                      "xtick.color": "white", "ytick.color": "white",
                      "text.color": "white", "axes.titlecolor": "white"})

print("✅ All imports successful")
print(f"   pandas {pd.__version__} | sklearn loaded | plotly loaded")


## 2. Data Loading & Cleaning

In [ ]:

CSV_PATH = "bitext-retail-ecommerce-llm-chatbot-training-dataset.csv"
df = pd.read_csv(CSV_PATH)

print("📦 Raw shape:", df.shape)
print("\n📋 Columns:", df.columns.tolist())
print("\n🔍 First 3 rows:")
display(df.head(3))


In [ ]:

# ── Missing values ──────────────────────────────────────────────────────
print("❌ Missing values per column:")
print(df.isnull().sum())

# ── Duplicate rows ──────────────────────────────────────────────────────
dupes = df.duplicated().sum()
print(f"\n🔁 Duplicate rows: {dupes}")
df.drop_duplicates(inplace=True)
print(f"   Shape after dedup: {df.shape}")

# ── Normalise text columns ──────────────────────────────────────────────
def clean_text(txt):
    txt = str(txt).strip()
    # remove multiple spaces
    txt = re.sub(r'\s+', ' ', txt)
    return txt

df['instruction'] = df['instruction'].apply(clean_text)
df['response']    = df['response'].apply(clean_text)
df['intent']      = df['intent'].str.strip().str.lower()
df['category']    = df['category'].str.strip().str.upper()

# ── Derived columns ─────────────────────────────────────────────────────
df['inst_len']    = df['instruction'].str.len()
df['inst_words']  = df['instruction'].str.split().str.len()
df['resp_len']    = df['response'].str.len()
df['resp_words']  = df['response'].str.split().str.len()
df['has_profanity'] = df['instruction'].str.contains(
    r'\b(fuck|shit|damn|crap|ass)\b', case=False, regex=True)

# ── Tag expansion ───────────────────────────────────────────────────────
TAG_MEANINGS = {
    'B': 'Basic',   'C': 'Contextual',  'E': 'Escalation',
    'I': 'Indirect','L': 'Long',        'M': 'Misspelled',
    'P': 'Polite',  'Q': 'Question',    'W': 'With_Profanity',
    'Z': 'Zigzag_syntax'
}
for tag, label in TAG_MEANINGS.items():
    df[f'tag_{label}'] = df['tags'].str.contains(tag, na=False)

print("\n✅ Cleaned & engineered. Shape:", df.shape)
display(df.head(3))


## 3. Exploratory Data Analysis (EDA)

In [ ]:

print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Total samples      : {len(df):,}")
print(f"Unique intents     : {df['intent'].nunique()}")
print(f"Unique categories  : {df['category'].nunique()}")
print(f"Rows with profanity: {df['has_profanity'].sum():,} ({df['has_profanity'].mean()*100:.1f}%)")
print(f"Avg instruction len: {df['inst_len'].mean():.1f} chars / {df['inst_words'].mean():.1f} words")
print(f"Avg response len   : {df['resp_len'].mean():.1f} chars / {df['resp_words'].mean():.1f} words")
print()
print("📊 Category distribution:")
display(df['category'].value_counts().to_frame())
print()
print("📊 Top 10 intents:")
display(df['intent'].value_counts().head(10).to_frame())


## 4. Matplotlib Static Visualisations

In [ ]:

# ── Fig 1: Category distribution bar chart ─────────────────────────────
cat_counts = df['category'].value_counts().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(cat_counts.index, cat_counts.values,
               color=plt.cm.plasma(np.linspace(0.1, 0.9, len(cat_counts))))
ax.set_xlabel("Number of samples", fontsize=12)
ax.set_title("Sample Count per Category", fontsize=15, fontweight='bold')
for bar, val in zip(bars, cat_counts.values):
    ax.text(val + 30, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig("plot_category_dist.png", dpi=130, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("✅ Fig 1 saved")


In [ ]:

# ── Fig 2: Top 20 intent distribution ──────────────────────────────────
intent_counts = df['intent'].value_counts().head(20)

fig, ax = plt.subplots(figsize=(13, 7))
colors = plt.cm.viridis(np.linspace(0.15, 0.9, len(intent_counts)))
ax.bar(intent_counts.index, intent_counts.values, color=colors)
ax.set_xticklabels(intent_counts.index, rotation=45, ha='right', fontsize=9)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Top 20 Intent Distribution", fontsize=15, fontweight='bold')
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("plot_intent_dist.png", dpi=130, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("✅ Fig 2 saved")


In [ ]:

# ── Fig 3: Instruction & response length distributions ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['inst_words'], bins=40, color='#e94560', edgecolor='white', linewidth=0.4)
axes[0].set_title("Instruction Word Count Distribution", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Words"); axes[0].set_ylabel("Frequency")
axes[0].axvline(df['inst_words'].mean(), color='cyan', linestyle='--',
                label=f"Mean={df['inst_words'].mean():.1f}")
axes[0].legend()

axes[1].hist(df['resp_words'], bins=40, color='#0f3460', edgecolor='white', linewidth=0.4)
axes[1].set_title("Response Word Count Distribution", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Words"); axes[1].set_ylabel("Frequency")
axes[1].axvline(df['resp_words'].mean(), color='orange', linestyle='--',
                label=f"Mean={df['resp_words'].mean():.1f}")
axes[1].legend()

plt.suptitle("Text Length Distributions", fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("plot_length_dist.png", dpi=130, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("✅ Fig 3 saved")


In [ ]:

# ── Fig 4: Tag frequency bar chart ─────────────────────────────────────
tag_cols = [c for c in df.columns if c.startswith('tag_')]
tag_sums = df[tag_cols].sum().sort_values(ascending=False)
tag_labels = [c.replace('tag_', '') for c in tag_sums.index]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(tag_labels, tag_sums.values,
              color=plt.cm.cool(np.linspace(0.1, 0.9, len(tag_labels))))
ax.set_ylabel("Frequency", fontsize=12)
ax.set_title("Tag Type Frequency Across Dataset", fontsize=14, fontweight='bold')
for bar, val in zip(bars, tag_sums.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}', ha='center', fontsize=8)
plt.tight_layout()
plt.savefig("plot_tag_freq.png", dpi=130, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("✅ Fig 4 saved")


In [ ]:

# ── Fig 5: Box-plot of instruction length by category ──────────────────
cats_order = df.groupby('category')['inst_words'].median().sort_values().index

fig, ax = plt.subplots(figsize=(14, 6))
data_by_cat = [df[df['category'] == c]['inst_words'].values for c in cats_order]
bp = ax.boxplot(data_by_cat, patch_artist=True, vert=True,
                labels=cats_order, showfliers=False)
colors = plt.cm.Set3(np.linspace(0, 1, len(cats_order)))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
ax.set_xlabel("Category", fontsize=12)
ax.set_ylabel("Instruction Word Count", fontsize=12)
ax.set_title("Instruction Length Distribution by Category", fontsize=14, fontweight='bold')
ax.set_xticklabels(cats_order, rotation=35, ha='right')
plt.tight_layout()
plt.savefig("plot_boxplot_cat.png", dpi=130, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("✅ Fig 5 saved")


In [ ]:

# ── Fig 6: Profanity rate by category ──────────────────────────────────
prof_rate = df.groupby('category')['has_profanity'].mean().sort_values(ascending=False)*100

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(prof_rate.index, prof_rate.values,
              color=['#ff6b6b' if v > prof_rate.mean() else '#4ecdc4'
                     for v in prof_rate.values])
ax.axhline(prof_rate.mean(), color='yellow', linestyle='--',
           label=f'Mean {prof_rate.mean():.1f}%')
ax.set_ylabel("Profanity Rate (%)", fontsize=12)
ax.set_title("Profanity Rate per Category", fontsize=14, fontweight='bold')
ax.set_xticklabels(prof_rate.index, rotation=35, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig("plot_profanity.png", dpi=130, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("✅ Fig 6 saved")


In [ ]:

# ── Fig 7: Heatmap — intents per category (top 30 intents) ─────────────
top30 = df['intent'].value_counts().head(30).index
heat_df = df[df['intent'].isin(top30)].groupby(
    ['category', 'intent']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(18, 8))
sns.heatmap(heat_df, annot=False, fmt='d', cmap='YlOrRd',
            linewidths=.3, ax=ax, cbar_kws={'label': 'Count'})
ax.set_title("Intent × Category Heatmap (top 30 intents)", fontsize=14, fontweight='bold')
ax.set_xlabel("Intent", fontsize=11)
ax.set_ylabel("Category", fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.savefig("plot_heatmap.png", dpi=130, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("✅ Fig 7 saved")


In [ ]:

# ── Fig 8: Pie chart — category share ──────────────────────────────────
cat_data = df['category'].value_counts()
explode = [0.04] * len(cat_data)

fig, ax = plt.subplots(figsize=(9, 9))
wedges, texts, autotexts = ax.pie(
    cat_data.values, labels=cat_data.index, autopct='%1.1f%%',
    explode=explode, colors=plt.cm.tab20.colors[:len(cat_data)],
    textprops={'color': 'white', 'fontsize': 9},
    pctdistance=0.82, startangle=140)
for at in autotexts:
    at.set_fontsize(8)
ax.set_title("Category Share", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("plot_category_pie.png", dpi=130, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("✅ Fig 8 saved")


In [ ]:

# ── Fig 9: Stacked bar — tag composition per category ──────────────────
tag_cols = [c for c in df.columns if c.startswith('tag_')]
tag_by_cat = df.groupby('category')[tag_cols].sum()
tag_by_cat.columns = [c.replace('tag_', '') for c in tag_by_cat.columns]
tag_by_cat_pct = tag_by_cat.div(tag_by_cat.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(14, 7))
tag_by_cat_pct.plot(kind='bar', stacked=True, ax=ax,
                    colormap='tab10', edgecolor='none', width=0.8)
ax.set_title("Tag Composition (%) by Category", fontsize=14, fontweight='bold')
ax.set_xlabel("Category")
ax.set_ylabel("Percentage (%)")
ax.legend(title="Tag", bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right')
plt.tight_layout()
plt.savefig("plot_tag_stack.png", dpi=130, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("✅ Fig 9 saved")


In [ ]:

# ── Fig 10: Scatter — instruction vs response length by category ─────────
fig, ax = plt.subplots(figsize=(11, 7))
cats = df['category'].unique()
cmap = plt.cm.get_cmap('tab20', len(cats))
for i, cat in enumerate(cats):
    sub = df[df['category'] == cat].sample(min(200, len(df[df['category']==cat])))
    ax.scatter(sub['inst_words'], sub['resp_words'], alpha=0.4,
               s=15, color=cmap(i), label=cat)
ax.set_xlabel("Instruction Word Count", fontsize=12)
ax.set_ylabel("Response Word Count", fontsize=12)
ax.set_title("Instruction vs Response Length by Category", fontsize=14, fontweight='bold')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7, markerscale=2)
plt.tight_layout()
plt.savefig("plot_scatter_len.png", dpi=130, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("✅ Fig 10 saved")


## 5. Plotly Interactive Dashboard

In [ ]:

# ── Plotly 1: Animated / interactive category bar ──────────────────────
cat_df = df['category'].value_counts().reset_index()
cat_df.columns = ['category', 'count']

fig = px.bar(cat_df, x='category', y='count', color='category',
             title='📦 Sample Count per Category (Interactive)',
             text='count', template='plotly_dark',
             color_discrete_sequence=px.colors.qualitative.Vivid)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, height=500,
                  xaxis_title='Category', yaxis_title='Count',
                  title_font_size=18)
fig.show()


In [ ]:

# ── Plotly 2: Treemap — category → intent hierarchy ────────────────────
tree_df = df.groupby(['category', 'intent']).size().reset_index(name='count')

fig = px.treemap(tree_df, path=['category', 'intent'], values='count',
                 title='🗂️ Intent Hierarchy Treemap (Category → Intent)',
                 template='plotly_dark',
                 color='count', color_continuous_scale='Plasma')
fig.update_layout(height=650, title_font_size=18)
fig.show()


In [ ]:

# ── Plotly 3: Sunburst chart ────────────────────────────────────────────
fig = px.sunburst(tree_df, path=['category', 'intent'], values='count',
                  title='☀️ Category → Intent Sunburst',
                  template='plotly_dark',
                  color='count', color_continuous_scale='Turbo')
fig.update_layout(height=650, title_font_size=18)
fig.show()


In [ ]:

# ── Plotly 4: Box plot — instruction words by category ─────────────────
fig = px.box(df, x='category', y='inst_words', color='category',
             title='📏 Instruction Word Count Distribution by Category',
             template='plotly_dark', points=False,
             color_discrete_sequence=px.colors.qualitative.Safe)
fig.update_layout(showlegend=False, height=520,
                  xaxis_title='Category', yaxis_title='Word Count',
                  xaxis_tickangle=-30, title_font_size=18)
fig.show()


In [ ]:

# ── Plotly 5: Violin plot — response length by category ─────────────────
fig = px.violin(df, x='category', y='resp_words', color='category',
                title='🎻 Response Word Count Violin by Category',
                template='plotly_dark', box=True,
                color_discrete_sequence=px.colors.qualitative.Alphabet)
fig.update_layout(showlegend=False, height=520,
                  xaxis_tickangle=-30, title_font_size=18)
fig.show()


In [ ]:

# ── Plotly 6: Scatter — inst vs resp length coloured by category ────────
sample_df = df.sample(3000, random_state=42)
fig = px.scatter(sample_df, x='inst_words', y='resp_words',
                 color='category', hover_data=['intent', 'instruction'],
                 title='🔵 Instruction vs Response Length (3 000 sample)',
                 template='plotly_dark', opacity=0.6, size_max=6,
                 color_discrete_sequence=px.colors.qualitative.Vivid)
fig.update_layout(height=550, title_font_size=18)
fig.show()


In [ ]:

# ── Plotly 7: Funnel chart — category volumes ──────────────────────────
funnel_df = df['category'].value_counts().reset_index()
funnel_df.columns = ['category', 'count']

fig = go.Figure(go.Funnel(
    y=funnel_df['category'],
    x=funnel_df['count'],
    textinfo='value+percent initial',
    marker=dict(color=px.colors.qualitative.Prism[:len(funnel_df)])
))
fig.update_layout(title='🔽 Category Volume Funnel',
                  template='plotly_dark', height=600, title_font_size=18)
fig.show()


In [ ]:

# ── Plotly 8: Heatmap — tag presence per category ──────────────────────
tag_cols = [c for c in df.columns if c.startswith('tag_')]
heat = df.groupby('category')[tag_cols].mean() * 100
heat.columns = [c.replace('tag_', '') for c in heat.columns]

fig = px.imshow(heat, text_auto='.1f', aspect='auto',
                title='🔥 Tag Presence Rate (%) per Category',
                template='plotly_dark', color_continuous_scale='Plasma',
                labels=dict(color='Rate (%)'))
fig.update_layout(height=500, title_font_size=18)
fig.show()


In [ ]:

# ── Plotly 9: Multi-metric summary dashboard ───────────────────────────
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        'Category Counts', 'Top 15 Intents',
        'Profanity Rate %', 'Avg Inst Words',
        'Avg Resp Words', 'Tag Distribution'
    ],
    specs=[[{"type": "bar"}, {"type": "bar"}, {"type": "bar"}],
           [{"type": "bar"}, {"type": "bar"}, {"type": "pie"}]]
)

cat_v = df['category'].value_counts()
fig.add_trace(go.Bar(x=cat_v.index, y=cat_v.values,
                     marker_color=px.colors.qualitative.Vivid,
                     showlegend=False), row=1, col=1)

top15i = df['intent'].value_counts().head(15)
fig.add_trace(go.Bar(x=top15i.index, y=top15i.values,
                     marker_color=px.colors.qualitative.Pastel,
                     showlegend=False), row=1, col=2)

prof = df.groupby('category')['has_profanity'].mean()*100
fig.add_trace(go.Bar(x=prof.index, y=prof.values,
                     marker_color='#ff6b6b', showlegend=False), row=1, col=3)

avg_inst = df.groupby('category')['inst_words'].mean()
fig.add_trace(go.Bar(x=avg_inst.index, y=avg_inst.values,
                     marker_color='#4ecdc4', showlegend=False), row=2, col=1)

avg_resp = df.groupby('category')['resp_words'].mean()
fig.add_trace(go.Bar(x=avg_resp.index, y=avg_resp.values,
                     marker_color='#ffa07a', showlegend=False), row=2, col=2)

tag_cols = [c for c in df.columns if c.startswith('tag_')]
tag_totals = df[tag_cols].sum()
fig.add_trace(go.Pie(labels=[c.replace('tag_','') for c in tag_totals.index],
                     values=tag_totals.values, showlegend=True), row=2, col=3)

fig.update_layout(height=800, template='plotly_dark',
                  title_text='📊 Retail Chatbot Dataset — Analytical Dashboard',
                  title_font_size=20)
fig.show()


In [ ]:

# ── Plotly 10: Parallel categories plot ────────────────────────────────
sample = df.sample(2000, random_state=1)
fig = px.parallel_categories(
    sample, dimensions=['category', 'intent'],
    color=pd.factorize(sample['category'])[0],
    color_continuous_scale=px.colors.sequential.Inferno,
    title='🔀 Category ↔ Intent Parallel Categories',
    template='plotly_dark'
)
fig.update_layout(height=600, title_font_size=18)
fig.show()


## 6. Feature Engineering & Preprocessing

In [ ]:

from sklearn.preprocessing import LabelEncoder

X = df['instruction'].values
y_intent   = df['intent'].values
y_category = df['category'].values

le_intent   = LabelEncoder()
le_category = LabelEncoder()
y_int_enc = le_intent.fit_transform(y_intent)
y_cat_enc = le_category.fit_transform(y_category)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_int_enc, test_size=0.2, random_state=42, stratify=y_int_enc)

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Intent classes : {len(le_intent.classes_)}")
print(f"Category classes: {len(le_category.classes_)}")


## 7. Intent Classification — TF-IDF + ML Models

In [ ]:

# ── Model 1: TF-IDF + Logistic Regression ──────────────────────────────
pipe_lr = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=30000,
                              sublinear_tf=True, min_df=2)),
    ('clf',   LogisticRegression(max_iter=1000, C=5, solver='saga',
                                  n_jobs=-1, random_state=42))
])
pipe_lr.fit(X_train, y_train)
y_pred_lr = pipe_lr.predict(X_test)

acc_lr = accuracy_score(y_test, y_pred_lr)
f1_lr  = f1_score(y_test, y_pred_lr, average='weighted')
print(f"🔵 Logistic Regression — Acc: {acc_lr:.4f} | F1: {f1_lr:.4f}")


In [ ]:

# ── Model 2: TF-IDF + Linear SVC ───────────────────────────────────────
pipe_svc = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=30000,
                               sublinear_tf=True, min_df=2)),
    ('clf',   LinearSVC(max_iter=3000, C=1.0, random_state=42))
])
pipe_svc.fit(X_train, y_train)
y_pred_svc = pipe_svc.predict(X_test)

acc_svc = accuracy_score(y_test, y_pred_svc)
f1_svc  = f1_score(y_test, y_pred_svc, average='weighted')
print(f"🟢 Linear SVC        — Acc: {acc_svc:.4f} | F1: {f1_svc:.4f}")


## 8. Model Evaluation & Visualisation

In [ ]:

# Classification report for best model (SVC)
best_pipe = pipe_svc if f1_svc >= f1_lr else pipe_lr
best_pred = y_pred_svc if f1_svc >= f1_lr else y_pred_lr
best_name = 'LinearSVC' if f1_svc >= f1_lr else 'LogisticRegression'

print(f"Best model: {best_name}")
print()
report = classification_report(y_test, best_pred,
                                target_names=le_intent.classes_)
print(report)


In [ ]:

# ── Confusion matrix (matplotlib) ──────────────────────────────────────
cm = confusion_matrix(y_test, best_pred)
n_classes = len(le_intent.classes_)

fig, ax = plt.subplots(figsize=(22, 18))
im = ax.imshow(cm, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=ax)
ax.set_xticks(range(n_classes))
ax.set_yticks(range(n_classes))
ax.set_xticklabels(le_intent.classes_, rotation=90, fontsize=7)
ax.set_yticklabels(le_intent.classes_, fontsize=7)
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=14, fontweight='bold')
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True', fontsize=11)
plt.tight_layout()
plt.savefig("plot_confusion_matrix.png", dpi=100, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("✅ Confusion matrix saved")


In [ ]:

# ── Per-intent F1 bar chart ─────────────────────────────────────────────
from sklearn.metrics import precision_recall_fscore_support
prec, rec, f1, _ = precision_recall_fscore_support(
    y_test, best_pred, labels=range(n_classes))

metrics_df = pd.DataFrame({
    'intent': le_intent.classes_,
    'precision': prec, 'recall': rec, 'f1': f1
}).sort_values('f1', ascending=False)

fig, ax = plt.subplots(figsize=(16, 7))
x = np.arange(len(metrics_df))
width = 0.28
ax.bar(x - width, metrics_df['precision'], width, label='Precision', color='#4ecdc4')
ax.bar(x,         metrics_df['recall'],    width, label='Recall',    color='#ffa07a')
ax.bar(x + width, metrics_df['f1'],        width, label='F1 Score',  color='#e94560')
ax.set_xticks(x)
ax.set_xticklabels(metrics_df['intent'], rotation=60, ha='right', fontsize=7)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Per-Intent Precision / Recall / F1', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig("plot_per_intent_metrics.png", dpi=130, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("✅ Per-intent metrics saved")


In [ ]:

# ── Plotly — interactive per-intent metrics ─────────────────────────────
fig = px.bar(metrics_df.melt(id_vars='intent', value_vars=['precision','recall','f1'],
                              var_name='metric', value_name='score'),
             x='intent', y='score', color='metric', barmode='group',
             title='📈 Per-Intent Precision / Recall / F1 (Interactive)',
             template='plotly_dark',
             color_discrete_sequence=px.colors.qualitative.Vivid)
fig.update_layout(height=550, xaxis_tickangle=-45, title_font_size=18)
fig.show()


In [ ]:

# ── Model comparison bar ────────────────────────────────────────────────
comp = pd.DataFrame({
    'Model': ['Logistic Regression', 'Linear SVC'],
    'Accuracy': [acc_lr, acc_svc],
    'F1 (weighted)': [f1_lr, f1_svc]
})

fig = px.bar(comp.melt(id_vars='Model'), x='Model', y='value', color='variable',
             barmode='group', title='🏆 Model Comparison', template='plotly_dark',
             color_discrete_sequence=['#4ecdc4', '#e94560'],
             text_auto='.4f')
fig.update_traces(textposition='outside')
fig.update_layout(height=420, yaxis_range=[0.85, 1.0], title_font_size=18)
fig.show()


## 9. Chatbot (TF-IDF Retrieval + ML Intent Classification)

In [ ]:

# ── Build the retrieval chatbot ─────────────────────────────────────────
from sklearn.metrics.pairwise import cosine_similarity

class RetailChatbot:
    """
    TF-IDF similarity retrieval chatbot backed by the training dataset.
    Predicts intent + category, then retrieves best-matching response.
    """
    def __init__(self, df, intent_pipe, le_intent):
        self.df = df.reset_index(drop=True)
        self.pipe = intent_pipe
        self.le   = le_intent
        
        print("Building TF-IDF index …", end=' ')
        self.vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=15000,
                                          sublinear_tf=True)
        self.tfidf_matrix = self.vectorizer.fit_transform(df['instruction'].values)
        print("done")
        self.history = []

    def predict_intent(self, text):
        enc = self.pipe.predict([text])[0]
        return self.le.inverse_transform([enc])[0]

    def retrieve_response(self, text, intent=None, top_k=5):
        q_vec = self.vectorizer.transform([text])
        sims  = cosine_similarity(q_vec, self.tfidf_matrix).flatten()
        
        if intent:
            mask = self.df['intent'] == intent
            masked_sims = sims.copy()
            masked_sims[~mask.values] = -1
            top_idx = masked_sims.argsort()[::-1][:top_k]
        else:
            top_idx = sims.argsort()[::-1][:top_k]
        
        best_idx = top_idx[0]
        return self.df.loc[best_idx, 'response'], sims[best_idx]

    def chat(self, user_input):
        intent  = self.predict_intent(user_input)
        cat_row = self.df[self.df['intent'] == intent]['category']
        category = cat_row.iloc[0] if not cat_row.empty else 'UNKNOWN'
        response, score = self.retrieve_response(user_input, intent)
        
        turn = {
            'user': user_input,
            'intent': intent,
            'category': category,
            'similarity': round(score, 4),
            'bot': response
        }
        self.history.append(turn)
        return turn

    def interactive(self):
        print("=" * 65)
        print("  🛒  Retail E-Commerce Chatbot  (type 'quit' to exit)")
        print("=" * 65)
        while True:
            try:
                user = input("\nYou: ").strip()
            except EOFError:
                break
            if not user:
                continue
            if user.lower() in ('quit', 'exit', 'bye'):
                print("Bot: Goodbye! 👋")
                break
            turn = self.chat(user)
            print(f"\n[Intent: {turn['intent']} | Category: {turn['category']} | Sim: {turn['similarity']:.3f}]")
            print(f"Bot: {turn['bot'][:400]}…")


# ── Instantiate ──────────────────────────────────────────────────────────
chatbot = RetailChatbot(df, best_pipe, le_intent)
print("\n✅ Chatbot ready!")


In [ ]:

# ── Demo: run a few test queries ────────────────────────────────────────
test_queries = [
    "I want to cancel my order",
    "Where is my delivery?",
    "How can I return a product?",
    "I need help with payment",
    "My item arrived damaged",
    "How do I open an account?"
]

print("\n--- CHATBOT DEMO ---\n")
for q in test_queries:
    turn = chatbot.chat(q)
    print(f"User    : {q}")
    print(f"Intent  : {turn['intent']}  |  Category: {turn['category']}  |  Sim: {turn['similarity']:.3f}")
    print(f"Bot     : {turn['bot'][:250]}…")
    print("-" * 70)


In [ ]:

# ── Uncomment to launch interactive terminal chatbot ─────────────────────
# chatbot.interactive()


## 10. t-SNE Embedding Visualisation

In [ ]:

# Reduce TF-IDF to 50 dims with SVD, then t-SNE to 2D
sample_idx = df.sample(2000, random_state=99).index
X_sample = best_pipe.named_steps['tfidf'].transform(df.loc[sample_idx, 'instruction'])
labels_sample = df.loc[sample_idx, 'category'].values

svd = TruncatedSVD(n_components=50, random_state=42)
X_50 = svd.fit_transform(X_sample)

tsne = TSNE(n_components=2, random_state=42, perplexity=40, n_iter=800)
X_2d = tsne.fit_transform(X_50)

tsne_df = pd.DataFrame({'x': X_2d[:,0], 'y': X_2d[:,1], 'category': labels_sample})

fig = px.scatter(tsne_df, x='x', y='y', color='category',
                 title='🔬 t-SNE of TF-IDF Embeddings — Coloured by Category (2 000 samples)',
                 template='plotly_dark', opacity=0.7,
                 color_discrete_sequence=px.colors.qualitative.Vivid)
fig.update_traces(marker_size=4)
fig.update_layout(height=620, title_font_size=18)
fig.show()
print("✅ t-SNE plot rendered")


## 11. Save Artefacts

In [ ]:

# Save best model pipeline
joblib.dump(best_pipe, f'intent_classifier_{best_name}.pkl')
joblib.dump(le_intent, 'label_encoder_intent.pkl')
joblib.dump(le_category, 'label_encoder_category.pkl')
df.to_csv('cleaned_dataset.csv', index=False)
print("✅ Saved:")
print(f"   intent_classifier_{best_name}.pkl")
print("   label_encoder_intent.pkl")
print("   label_encoder_category.pkl")
print("   cleaned_dataset.csv")
